In [1]:
%pip install seleniumbase

Note: you may need to restart the kernel to use updated packages.


In [12]:
from seleniumbase import Driver
from bs4 import BeautifulSoup
import time
import random
import csv
import re
import os

def scrape_mobile_de_details(max_pages=5):
    print("Starte getarnten Browser...")
    driver = Driver(uc=True, incognito=True)
    
    # TRAGE HIER DEINE GEWÜNSCHTE URL EIN (Basis-URL ohne Seitenzahl)
    base_search_url = "https://suchen.mobile.de/fahrzeuge/search.html?cn=DE&dam=false&isSearchRequest=true&ms=17200%3B%3B16%3B&od=up&ref=srp&refId=c0d9ecaa-2e9d-52e5-6f6c-f7ea6ddbf3b2&s=Car&sb=rel&st=DEALER&vc=Car" 
    
    filename = 'mobile_de_erweitert_Sklasse.csv'
    fieldnames = [
        'Titel', 'Preis', 'Kilometerstand', 'Erstzulassung', 
        'PS', 'Getriebe', 'Kraftstoff', 'Fahrzeughalter', 
        'Standort', 'URL', 'Ausstattung', 'Beschreibung'
    ]

    # CSV initialisieren (Header schreiben, falls Datei neu ist)
    if not os.path.exists(filename):
        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()

    all_car_links = []

    try:
        # --- 1. LINKS ÜBER MEHRERE SEITEN SAMMELN ---
        for page in range(1, max_pages + 1):
            current_search_url = f"{base_search_url}&pageNumber={page}"
            print(f"\n--- Lade Übersichtsseite {page} ---")
            
            driver.uc_open_with_reconnect(current_search_url, reconnect_time=5)
            
            if page == 1:
                print("Warte 15 Sekunden auf der ersten Seite (evtl. Captcha/Cookies)...")
                time.sleep(15)
            else:
                time.sleep(random.uniform(5, 8))
            
            driver.execute_script("window.scrollBy(0, 1500);")
            time.sleep(3)
            
            soup_overview = BeautifulSoup(driver.page_source, 'html.parser')
            page_links = []
            
            for a_tag in soup_overview.find_all('a', href=re.compile(r'/fahrzeuge/details\.html')):
                href = a_tag.get('href')
                if href.startswith('/'):
                    href = "https://suchen.mobile.de" + href
                # URL bereinigen, falls Parameter angehängt sind, um Duplikate zu vermeiden
                clean_href = href.split('?')[0] + "?" + href.split('?')[1].split('&')[0] if '?' in href else href
                if clean_href not in all_car_links:
                    page_links.append(clean_href)
                    
            if not page_links:
                print("Keine weiteren Links gefunden. Beende das Sammeln der Links.")
                break
                
            all_car_links.extend(page_links)
            print(f"Es wurden {len(page_links)} Auto-Links auf Seite {page} gefunden. (Gesamt: {len(all_car_links)})")

        # LIMIT FÜR DEN TESTLAUF (Entferne das, wenn du alle gesammelten Links scrapen willst)
        all_car_links = all_car_links[:200] 
        print(f"\nBesuche nun {len(all_car_links)} Autos im Detail...")

        # --- 2. DETAILSEITEN BESUCHEN & DIREKT SPEICHERN ---
        for index, link in enumerate(all_car_links, 1):
            try:
                print(f"[{index}/{len(all_car_links)}] Öffne Auto...")
                driver.uc_open_with_reconnect(link, reconnect_time=4)
                time.sleep(random.uniform(4.0, 6.0))
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(1.0)
                
                # Klicke alle "Mehr anzeigen" Buttons auf der Seite per JavaScript
                try:
                    mehr_anzeigen_buttons = driver.find_elements('xpath', "//*[contains(text(), 'Mehr anzeigen')]")
                    for btn in mehr_anzeigen_buttons:
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.5) 
                except Exception:
                    pass 
                
                soup = BeautifulSoup(driver.page_source, 'html.parser')
                page_text = soup.get_text(separator=' ', strip=True)
                
                # -- EXAKTER TITEL / MODELL (3-Stufen-Methode) --
                title = 'N/A'
                
                # Stufe 1: Meta-Tag (Sehr zuverlässig)
                meta_title = soup.find('meta', property='og:title')
                if meta_title and meta_title.get('content'):
                    title = meta_title.get('content')
                    # Entfernt eventuell angehängte Preisangaben wie " für 45.000 €"
                    title = re.sub(r'\s*für\s*[\d\.,]+\s*€.*', '', title).strip()
                    
                # Stufe 2: Interne IDs oder H1
                if title == 'N/A' or not title:
                    title_elem = soup.find(attrs={"data-testid": "listing-title"}) or soup.find(id="ad-title") or soup.find('h1')
                    if title_elem:
                        title = title_elem.get_text(separator=' ', strip=True)
                
                # Stufe 3: Webseiten-Tab-Titel
                if title == 'N/A' or not title:
                    page_title = soup.find('title')
                    if page_title:
                        # Schneidet Zusätze ab (Bsp: "Mercedes C 220 für 40.000€ | mobile.de")
                        title = page_title.text.split('für')[0].split('|')[0].strip()

                # -- BASISDATEN --
                
                # KAUFPREIS (Keine Jahres-/Monatsraten)
                price = 'N/A'
                # 1. Versuch: Gezielter Griff nach dem Hauptpreis-Element
                price_elem = soup.find(attrs={"data-testid": "prime-price"})
                if price_elem:
                    price = price_elem.get_text(strip=True)
                    
                # 2. Versuch: Text-Suche, aber gefiltert!
                if price == 'N/A' or not price:
                    # Sucht nach Zahlen wie 12.345 €
                    price_tags = soup.find_all(string=re.compile(r'\d{1,3}(?:\.\d{3})*\s*€'))
                    for tag in price_tags:
                        text = tag.strip()
                        # Filtere Energiekosten (/Jahr) und Leasing/Finanzierung (mtl.) heraus
                        if not any(bad_word in text.lower() for bad_word in ['/jahr', 'mtl', 'monat', 'ab ']):
                            match = re.search(r'(\d{1,3}(?:\.\d{3})*\s*€)', text)
                            if match:
                                price = match.group(1)
                                break
                        
                km_match = re.search(r'([\d\.]+)\s*km', page_text, re.IGNORECASE)
                mileage = km_match.group(0) if km_match else 'N/A'
                    
                ez_match = re.search(r'(?:EZ|Erstzulassung)[\s:]*(\d{2}/\d{4})', page_text, re.IGNORECASE)
                if not ez_match:
                    ez_match = re.search(r'\b(0[1-9]|1[0-2])/(19|20)\d{2}\b', page_text)
                registration = ez_match.group(0) if ez_match else 'N/A'

                ps_match = re.search(r'\((\d+)\s*PS\)', page_text)
                if not ps_match:
                    ps_match = re.search(r'(\d+)\s*PS', page_text)
                ps = f"{ps_match.group(1)} PS" if ps_match else 'N/A'

                getriebe = 'Automatik' if re.search(r'\bAutomatik\b', page_text, re.IGNORECASE) else ('Schaltgetriebe' if re.search(r'\bSchaltgetriebe\b', page_text, re.IGNORECASE) else 'N/A')
                
                kraftstoff = 'N/A'
                for k in ['Benzin', 'Diesel', 'Elektro', 'Hybrid', 'Autogas (LPG)', 'Erdgas (CNG)']:
                    if re.search(rf'\b{re.escape(k)}\b', page_text, re.IGNORECASE):
                        kraftstoff = k
                        break

                halter_match = re.search(r'Fahrzeughalter(?:[^0-9]{1,15})(\d+)', page_text, re.IGNORECASE)
                halter = halter_match.group(1) if halter_match else 'N/A'

                # -- STANDORT (Verbesserte Logik) --
                standort = 'N/A'
                loc_match = re.search(r'DE-([0-9]{5})\s+([A-ZÄÖÜ][a-zA-ZäöüßÄÖÜ\-\s]+?)(?=\s*\||\s*$)', page_text)
                if loc_match:
                    standort = f"{loc_match.group(1)} {loc_match.group(2).strip()}"
                else:
                    fallback = re.search(r'\b([0-9]{5})\s+([A-ZÄÖÜ][a-zäöüß]+)\b', page_text)
                    if fallback and fallback.group(2) not in ["Verfügbarkeit", "Kilometer", "Euro"]:
                        standort = f"{fallback.group(1)} {fallback.group(2)}"

                # -- 1. AUSSTATTUNG (GEFILTERT) --
                ausstattung_liste = []
                ausstattung_heading = soup.find(lambda tag: tag.name in ['h2', 'h3', 'div'] and tag.text.strip() == 'Ausstattung')
                
                if ausstattung_heading:
                    for element in ausstattung_heading.find_all_next(string=True):
                        text = element.strip()
                        if text in ["Fahrzeugbeschreibung laut Anbieter", "Fahrzeugstandort", "Über diesen Händler", "Standort", "Preis"]:
                            break
                        if text and text not in ["Ausstattung", "Mehr anzeigen", "Weniger anzeigen", "✓"]:
                            if text not in ausstattung_liste and len(text) > 2:
                                ausstattung_liste.append(text)
                
                ausstattung_str = ", ".join(ausstattung_liste) if ausstattung_liste else "N/A"

                # -- 2. FAHRZEUGBESCHREIBUNG LAUT ANBIETER (GEFILTERT) --
                beschreibung_text = 'N/A'
                beschr_heading = soup.find(lambda tag: tag.name in ['h2', 'h3', 'div'] and tag.text.strip() == 'Fahrzeugbeschreibung laut Anbieter')
                
                if beschr_heading:
                    beschr_strings = []
                    for element in beschr_heading.find_all_next(string=True):
                        text = element.strip()
                        if text in ["Über diesen Händler", "Standort", "Händler", "Preis", "Finanzierung", "Ähnliche Fahrzeuge"]:
                            break
                        if text and text not in ["Fahrzeugbeschreibung laut Anbieter", "Mehr anzeigen", "Weniger anzeigen"]:
                            if len(text) > 1:
                                beschr_strings.append(text)
                    
                    clean_strings = []
                    for s in beschr_strings:
                        if s not in clean_strings:
                            clean_strings.append(s)
                            
                    beschreibung_text = " | ".join(clean_strings) if clean_strings else "N/A"
                
                # --- DATEN DIREKT IN CSV SPEICHERN ---
                car_data = {
                    'Titel': title,
                    'Preis': price,
                    'Kilometerstand': mileage,
                    'Erstzulassung': registration,
                    'PS': ps,
                    'Getriebe': getriebe,
                    'Kraftstoff': kraftstoff,
                    'Fahrzeughalter': halter,
                    'Standort': standort,
                    'URL': link,
                    'Ausstattung': ausstattung_str,
                    'Beschreibung': beschreibung_text
                }
                
                with open(filename, 'a', newline='', encoding='utf-8') as csvfile:
                    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                    writer.writerow(car_data)

                print(f"   -> Erfolg & Gespeichert: {title[:40]}... | {price} | Standort: {standort}")
                
                
            except Exception as e:
                print(f"   -> Fehler bei diesem Auto übersprungen: {e}")
                continue # Macht mit dem nächsten Auto weiter

    except Exception as e:
        print(f"Ein genereller Fehler ist aufgetreten: {e}")
    finally:
        driver.quit()
        print(f"\nScraping beendet! Alle gesammelten Daten sind in '{filename}'.")

# --- Ausführung ---
# Du kannst die Anzahl der durchsuchten Seiten hier anpassen (z.B. max_pages=10)
scrape_mobile_de_details(max_pages=7)

Starte getarnten Browser...

--- Lade Übersichtsseite 1 ---
Warte 15 Sekunden auf der ersten Seite (evtl. Captcha/Cookies)...
Es wurden 24 Auto-Links auf Seite 1 gefunden. (Gesamt: 24)

--- Lade Übersichtsseite 2 ---
Es wurden 20 Auto-Links auf Seite 2 gefunden. (Gesamt: 44)

--- Lade Übersichtsseite 3 ---
Es wurden 20 Auto-Links auf Seite 3 gefunden. (Gesamt: 64)

--- Lade Übersichtsseite 4 ---
Es wurden 20 Auto-Links auf Seite 4 gefunden. (Gesamt: 84)

--- Lade Übersichtsseite 5 ---
Es wurden 20 Auto-Links auf Seite 5 gefunden. (Gesamt: 104)

--- Lade Übersichtsseite 6 ---
Es wurden 20 Auto-Links auf Seite 6 gefunden. (Gesamt: 124)

--- Lade Übersichtsseite 7 ---
Es wurden 20 Auto-Links auf Seite 7 gefunden. (Gesamt: 144)

Besuche nun 144 Autos im Detail...
[1/144] Öffne Auto...
   -> Erfolg & Gespeichert: Mercedes-Benz S 63 AMG... | 65.900 € | Standort: 45699 Herten
[2/144] Öffne Auto...
   -> Erfolg & Gespeichert: Mercedes-Benz S 350... | 35.999 € | Standort: 14513 Teltow
[3/144] Ö